# COSC726 · Lab 6 — Plan, Critique, Re-plan
### Real model · self-contained · four goals, four different failures

**Week 7 · ~2.5 hours · Colab or local**

For three weeks your agent decided **one step at a time**. Thought and action
fired in the same turn, so there was never a moment at which a plan existed
to be checked before anything ran.

Today it commits to a plan first. That buys you a checkpoint before
irreversible actions and costs you the freedom to adapt. **Your job is to
measure that trade, not to assert it.**

| Part | You build | Kind |
|---|---|---|
| 1 | The world and the plan contract | given — read it |
| 2 | Validate a plan before running it | **Task 1** |
| 3 | Execute, with the Week 4 gates still in front | **Task 2** |
| 4 | The loop: plan → execute → critique → re-plan | **Task 3** |
| 5 | Measure across four goals | **Task 4** |
| 6 | The exercises | assessed |

### The one thing to read before you start

Reflection is **not** a universal repair. It works when the failure is
diagnosable from the output — a missing step, a wrong argument. It does
nothing at all when the failure is **structural**: a tool that does not
exist, a permission always denied, a threshold not met.

**Goal G4 is structural.** Watch what your loop does with it.


## Part 0 — Setup

Ollama is the default: free, no account, no rate limit. For the hosted path
on Colab, use the **key icon** in the left sidebar — never paste a key into
a cell.

In [ ]:
# @title Setup — install, credentials, and lab7_kit on your PC { display-mode: "form" }
!pip -q install "openai>=1.40" "pydantic>=2.7" 2>&1 | tail -1

import os, subprocess, time, urllib.request

# Colab has no Ollama; install and start one unless you are pointing at a
# hosted provider instead.
def ollama_up(url="http://localhost:11434"):
    try:
        urllib.request.urlopen(url, timeout=2); return True
    except Exception:
        return False

if os.getenv("LLM_PROVIDER", "ollama") == "ollama" and not ollama_up():
    print("installing Ollama ...")
    !curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -1
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):
        if ollama_up(): break
        time.sleep(1)
    !ollama pull qwen2.5:7b 2>&1 | tail -1

# Colab secrets, if present
try:
    from google.colab import userdata
    for k in ("OPENAI_API_KEY",):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
except ImportError:
    pass

print("ollama running:", ollama_up())

installing Ollama ...
  - Arch: sudo pacman -S zstd


FileNotFoundError: [Errno 2] No such file or directory: 'ollama'

In [13]:

# @title Setup — install, credentials, and lab7_kit on Colab  { display-mode: "form" }
!pip -q install "openai>=1.40" "pydantic>=2.7" 2>&1 | tail -1

import os
import platform
import subprocess
import time
import urllib.request
import urllib.error
import json
import shutil


# ============================================================
# 1. Install Python dependencies
# ============================================================

print("=== 1. Installing Python dependencies ===")

subprocess.run(
    ["pip", "install", "-q", "-U", "chromadb"],
    check=True
)

print("✓ ChromaDB installed")


# ============================================================
# 2. Detect Colab architecture
# ============================================================

print("\n=== 2. Detecting system architecture ===")

machine = platform.machine().lower()

print("Detected architecture:", machine)

if machine in ("x86_64", "amd64"):
    ollama_arch = "amd64"

elif machine in ("aarch64", "arm64"):
    ollama_arch = "arm64"

else:
    raise RuntimeError(
        f"Unsupported architecture: {machine}"
    )

print("✓ Using Ollama architecture:", ollama_arch)


# ============================================================
# 3. Install system dependency: zstd
# ============================================================

print("\n=== 3. Installing zstd ===")

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True
)

print("✓ zstd installed")


# ============================================================
# 4. Remove broken previous Ollama installation
# ============================================================

print("\n=== 4. Cleaning previous Ollama installation ===")

possible_paths = [
    "/usr/local/bin/ollama",
    "/usr/bin/ollama"
]

for path in possible_paths:
    if os.path.isfile(path):
        try:
            result = subprocess.run(
                [path, "--version"],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=5
            )

            if result.returncode != 0:
                print("Removing broken Ollama:", path)
                os.remove(path)

        except (OSError, subprocess.SubprocessError):
            print("Removing invalid Ollama:", path)
            os.remove(path)


# Remove previous Ollama libraries if present
if os.path.isdir("/usr/lib/ollama"):
    print("Removing previous Ollama libraries...")
    shutil.rmtree(
        "/usr/lib/ollama",
        ignore_errors=True
    )


# ============================================================
# 5. Download official Ollama Linux archive
# ============================================================

print("\n=== 5. Downloading Ollama ===")

OLLAMA_DOWNLOAD = (
    f"https://ollama.com/download/"
    f"ollama-linux-{ollama_arch}.tar.zst"
)

ARCHIVE_PATH = f"/tmp/ollama-linux-{ollama_arch}.tar.zst"

print("Download URL:")
print(OLLAMA_DOWNLOAD)

download_result = subprocess.run(
    [
        "curl",
        "-fL",
        "--retry", "3",
        "--retry-delay", "2",
        "-o", ARCHIVE_PATH,
        OLLAMA_DOWNLOAD
    ]
)

if download_result.returncode != 0:
    raise RuntimeError(
        "Failed to download Ollama archive."
    )

if not os.path.exists(ARCHIVE_PATH):
    raise RuntimeError(
        "Ollama archive was not downloaded."
    )

archive_size = os.path.getsize(ARCHIVE_PATH)

print(
    "✓ Downloaded:",
    round(archive_size / (1024**3), 2),
    "GB"
)


# ============================================================
# 6. Extract Ollama into /usr
# ============================================================

print("\n=== 6. Extracting Ollama ===")

extract_result = subprocess.run(
    [
        "tar",
        "--zstd",
        "-xf",
        ARCHIVE_PATH,
        "-C",
        "/usr"
    ]
)

if extract_result.returncode != 0:
    raise RuntimeError(
        "Failed to extract Ollama."
    )

print("✓ Ollama extracted")


# ============================================================
# 7. Find Ollama executable
# ============================================================

ollama_path = shutil.which("ollama")

if ollama_path is None:

    candidates = [
        "/usr/bin/ollama",
        "/usr/local/bin/ollama"
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            ollama_path = candidate
            break


if ollama_path is None:
    raise RuntimeError(
        "Ollama executable could not be found."
    )


print("\nOllama executable:", ollama_path)


# ============================================================
# 8. Verify Ollama executable
# ============================================================

print("\n=== 7. Verifying Ollama ===")

try:

    version = subprocess.run(
        [ollama_path, "--version"],
        capture_output=True,
        text=True,
        timeout=15
    )

except OSError as e:

    raise RuntimeError(
        f"Ollama executable exists but cannot run: {e}"
    )


print(
    version.stdout.strip()
    or version.stderr.strip()
)


if version.returncode != 0:
    raise RuntimeError(
        "Ollama executable failed verification."
    )


print("✓ Ollama binary works")


# ============================================================
# 9. Helper to check Ollama API
# ============================================================

OLLAMA_URL = "http://127.0.0.1:11434"


def ollama_up():

    try:

        with urllib.request.urlopen(
            OLLAMA_URL,
            timeout=2
        ) as response:

            return response.status == 200

    except Exception:
        return False


# ============================================================
# 10. Start Ollama server
# ============================================================

print("\n=== 8. Starting Ollama server ===")


if not ollama_up():

    log_path = "/tmp/ollama.log"

    log_file = open(
        log_path,
        "w"
    )

    env = os.environ.copy()

    # Important for Colab
    env["OLLAMA_HOST"] = "127.0.0.1:11434"

    ollama_process = subprocess.Popen(
        [ollama_path, "serve"],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env
    )

    print("Waiting for Ollama API...")

    for i in range(60):

        if ollama_up():
            break

        if ollama_process.poll() is not None:

            log_file.close()

            print("\n--- Ollama log ---")

            if os.path.exists(log_path):

                with open(log_path) as f:
                    print(f.read())

            raise RuntimeError(
                "Ollama server stopped unexpectedly."
            )

        time.sleep(1)


if not ollama_up():

    print("\n--- Ollama log ---")

    if os.path.exists("/tmp/ollama.log"):

        with open("/tmp/ollama.log") as f:
            print(f.read())

    raise RuntimeError(
        "Ollama API did not start."
    )


print("✓ Ollama server running")
print("✓ API:", OLLAMA_URL)


# ============================================================
# 11. Pull qwen2.5:7b model
# ============================================================

MODEL_NAME = "qwen2.5:3b"

print(
    f"\n=== 9. Pulling {MODEL_NAME} ==="
)


pull_result = subprocess.run(
    [
        ollama_path,
        "pull",
        MODEL_NAME
    ]
)


if pull_result.returncode != 0:

    raise RuntimeError(
        f"Failed to pull {MODEL_NAME}"
    )


print(
    f"✓ {MODEL_NAME} ready"
)


# ============================================================
# 12. Show installed models
# ============================================================

print("\n=== 10. Installed models ===")

subprocess.run(
    [
        ollama_path,
        "list"
    ],
    check=False
)




# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 60)
print("LAB 6 SETUP COMPLETE")
print("=" * 60)

print(
    "Architecture       :",
    ollama_arch
)

print(
    "Ollama executable :",
    ollama_path
)

print(
    "Ollama API        :",
    OLLAMA_URL
)

print(
    " model   :",
    MODEL_NAME
)


print("=" * 60)

=== 1. Installing Python dependencies ===
✓ ChromaDB installed

=== 2. Detecting system architecture ===
Detected architecture: x86_64
✓ Using Ollama architecture: amd64

=== 3. Installing zstd ===
✓ zstd installed

=== 4. Cleaning previous Ollama installation ===
Removing previous Ollama libraries...

=== 5. Downloading Ollama ===
Download URL:
https://ollama.com/download/ollama-linux-amd64.tar.zst
✓ Downloaded: 1.33 GB

=== 6. Extracting Ollama ===
✓ Ollama extracted

Ollama executable: /usr/bin/ollama

=== 7. Verifying Ollama ===
✓ Ollama binary works

=== 8. Starting Ollama server ===
Waiting for Ollama API...
✓ Ollama server running
✓ API: http://127.0.0.1:11434

=== 9. Pulling qwen2.5:3b ===
✓ qwen2.5:3b ready

=== 10. Installed models ===

LAB 6 SETUP COMPLETE
Architecture       : amd64
Ollama executable : /usr/bin/ollama
Ollama API        : http://127.0.0.1:11434
 model   : qwen2.5:3b


In [14]:
# @title Write lab7_kit.py into the runtime  { display-mode: "form" }
os.environ["OLLAMA_MODEL"] = "qwen2.5:3b"
kit_source = r'''"""
COSC726 Lab 6 — planning, reflection and re-planning (support module)
=====================================================================
Real model, no mocks, no other lab required.

    pip install openai pydantic
    ollama pull qwen2.5:7b && ollama serve
    python layla_planner_solution.py

What changes this week
----------------------
Weeks 4-6 built an agent that decides ONE step at a time. That is ReAct, and
its defining property is that Thought and Action fire in the same turn --
there is no point at which a plan exists to be inspected before anything
runs.

This week the agent commits to a plan first. That buys you a checkpoint
before irreversible actions, and it costs you the ability to adapt freely.
The lab is about measuring that trade, not asserting it.

    Plan  ->  Execute  ->  Critique  ->  Re-plan  ->  (stop)

Public API
----------
    GOALS                 four multi-step requests, with gold step sets
    Step, Plan            the plan contract (Pydantic, validated)
    TOOLS                 the Week 4 tool set, with tiers and gates
    PlanTrace             every plan version, every step, every critique
    Critique              the critic's verdict, as a type
    detect_oscillation()  the loop detector
    goal_drift()          did the plan stop serving the goal?
    score_plan()          plan quality against the gold step set
    PLANNER_SYSTEM / CRITIC_SYSTEM     prompt scaffolds

The finding this lab exists to produce
--------------------------------------
Reflexion improves things a lot when the failure is *diagnosable from the
output* -- reported gains on coding benchmarks run to roughly twenty points
over a single attempt. It does nothing at all when the failure is
STRUCTURAL: a missing permission, a tool that does not exist, a policy
threshold not met. Reflecting harder on "permission denied" produces a more
eloquent way of being denied.

Exercise 3 makes that concrete. Watch the critic loop three times on a
problem no amount of reflection can solve, then decide what the agent should
have done instead.
"""
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Callable, Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

__all__ = [
    "ORDERS", "KNOWN_IDS", "THRESHOLD_DAYS", "Tier", "TOOLS", "ToolSpec",
    "Step", "Plan", "Critique", "PlanTrace", "PlanVersion", "StepResult",
    "GOALS", "Goal", "detect_oscillation", "goal_drift", "score_plan",
    "PLANNER_SYSTEM", "CRITIC_SYSTEM", "make_client", "MODEL",
]

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
MODEL = (os.getenv("OLLAMA_MODEL", "qwen2.5:7b") if PROVIDER == "ollama"
         else os.getenv("OPENAI_MODEL", "gpt-4o-mini-2024-07-18"))


def make_client():
    """The Week 2 seam. Ollama and OpenAI speak the same dialect."""
    from openai import OpenAI
    if PROVIDER == "ollama":
        base = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
        return OpenAI(base_url=f"{base}/v1", api_key="ollama")
    if not os.getenv("OPENAI_API_KEY"):
        raise SystemExit("OPENAI_API_KEY is not set (or use LLM_PROVIDER=ollama)")
    return OpenAI()


# ---------------------------------------------------------------------------
# 1. The world — the same Northwind, so nothing here is new to learn
# ---------------------------------------------------------------------------

ORDERS: dict[str, dict[str, Any]] = {
    "A1032": {"promised": "Tue", "eta": "Fri", "days_late": 3,
              "status": "delayed_at_depot", "value": 84.00},
    "A1044": {"promised": "Mon", "eta": "Mon", "days_late": 0,
              "status": "out_for_delivery", "value": 31.50},
    "A1080": {"promised": "Thu", "eta": "Fri", "days_late": 1,
              "status": "delayed_in_transit", "value": 126.00},
    "A1091": {"promised": "Mon", "eta": "Fri", "days_late": 14,
              "status": "delayed_at_depot", "value": 59.99},
}
KNOWN_IDS = set(ORDERS)
THRESHOLD_DAYS = 3
APPROVALS: dict[str, dict] = {}


def ok(**f):
    return {"ok": True, **f}


def err(code, **f):
    return {"ok": False, "error": code, **f}


class Tier(str, Enum):
    READ = "read"
    WRITE = "write"
    CONSEQUENTIAL = "consequential"


def track_order(order_id: str) -> dict:
    row = ORDERS.get(order_id)
    if row is None:
        return err("order_not_found", order_id=order_id)
    return ok(order_id=order_id, **row)


def get_policy() -> dict:
    return ok(threshold_days=THRESHOLD_DAYS, credit_percent=10,
              text=f"Orders {THRESHOLD_DAYS}+ working days late qualify for "
                   "a 10% credit, which requires supervisor approval. "
                   "Billing disputes are handled by the billing team, never "
                   "by support.")


def check_address_changeable(order_id: str) -> dict:
    row = ORDERS.get(order_id)
    if row is None:
        return err("order_not_found", order_id=order_id)
    changeable = row["status"] != "out_for_delivery"
    return ok(order_id=order_id, changeable=changeable,
              reason=("still at depot" if changeable
                      else "already with the courier; customer must arrange "
                           "redelivery"))


def request_approval(order_id: str, amount_percent: int) -> dict:
    """CONSEQUENTIAL. Creates a PENDING record. Applies nothing."""
    if order_id not in ORDERS:
        return err("order_not_found", order_id=order_id)
    ref = f"APR-{2048 + len(APPROVALS)}"
    APPROVALS[ref] = {"order_id": order_id, "state": "pending"}
    return ok(approval_ref=ref, state="pending", account_changed=False)


def escalate_to_billing(order_id: str, description: str) -> dict:
    return ok(escalated=True, team="billing", order_id=order_id,
              description=description)


def escalate_to_human(reason: str) -> dict:
    return ok(escalated=True, reason=reason)


@dataclass(frozen=True)
class ToolSpec:
    fn: Callable[..., dict]
    tier: Tier
    description: str
    args: list[str]


TOOLS: dict[str, ToolSpec] = {
    "track_order": ToolSpec(
        track_order, Tier.READ,
        "Look up ONE order: status, days_late, value. Read-only.",
        ["order_id"]),
    "get_policy": ToolSpec(
        get_policy, Tier.READ,
        "Return the late-delivery policy and its threshold. Read-only.",
        []),
    "check_address_changeable": ToolSpec(
        check_address_changeable, Tier.READ,
        "Can this order's delivery address still be changed? Read-only.",
        ["order_id"]),
    "request_approval": ToolSpec(
        request_approval, Tier.CONSEQUENTIAL,
        "Create a PENDING credit approval. Applies nothing.",
        ["order_id", "amount_percent"]),
    "escalate_to_billing": ToolSpec(
        escalate_to_billing, Tier.WRITE,
        "Hand a payment dispute to the billing team.",
        ["order_id", "description"]),
    "escalate_to_human": ToolSpec(
        escalate_to_human, Tier.WRITE,
        "Hand the whole case to a person, with context.",
        ["reason"]),
}


# ---------------------------------------------------------------------------
# 2. The plan contract
# ---------------------------------------------------------------------------

class Step(BaseModel):
    """One step of a plan. Note it is a PROPOSAL: nothing has run yet."""
    model_config = ConfigDict(extra="forbid")
    n: int = Field(ge=1, le=12)
    tool: str
    args: dict = Field(default_factory=dict)
    why: str = Field(max_length=200,
                     description="What this step establishes, in one line.")


class Plan(BaseModel):
    """A whole plan, produced before anything executes.

    THIS is what Plan-and-Execute buys you over ReAct: an artefact that
    exists before any action, and can therefore be inspected, validated,
    priced or shown to a human. ReAct has no such moment -- its Thought and
    Action fire in the same turn.
    """
    model_config = ConfigDict(extra="forbid")
    goal_restated: str = Field(max_length=300)
    steps: list[Step] = Field(min_length=1, max_length=12)

    def signature(self) -> str:
        """Identity of the plan's shape, for oscillation detection."""
        return "|".join(f"{s.tool}({json.dumps(s.args, sort_keys=True)})"
                        for s in self.steps)


class Critique(BaseModel):
    """The critic's verdict. A type, not a paragraph.

    `structural` is the field that matters. A structural failure is one that
    re-planning CANNOT fix: a missing permission, a tool that does not
    exist, a policy threshold not met. Reflecting harder on those produces a
    more eloquent way of being stuck.
    """
    model_config = ConfigDict(extra="forbid")
    goal_met: bool
    problems: list[str] = Field(default_factory=list, max_length=5)
    structural: bool = Field(
        default=False,
        description="True when no re-plan can fix this and a human is needed.")
    revise: bool = False


# ---------------------------------------------------------------------------
# 3. The trace — instrumentation is the deliverable
# ---------------------------------------------------------------------------

@dataclass
class StepResult:
    n: int
    tool: str
    args: dict
    tier: str | None
    ok: bool
    error: str | None = None
    observation: dict = field(default_factory=dict)


@dataclass
class PlanVersion:
    version: int
    plan: Plan | None
    results: list[StepResult] = field(default_factory=list)
    critique: Critique | None = None
    tokens: int = 0
    invalid_reason: str | None = None

    @property
    def signature(self) -> str:
        return self.plan.signature() if self.plan else "(invalid)"


@dataclass
class PlanTrace:
    goal_id: str
    versions: list[PlanVersion] = field(default_factory=list)
    stop_reason: str = ""
    answer: str | None = None

    @property
    def total_tokens(self) -> int:
        return sum(v.tokens for v in self.versions)

    @property
    def replans(self) -> int:
        return max(len(self.versions) - 1, 0)

    def render(self) -> str:
        out = [f"goal {self.goal_id}"]
        for v in self.versions:
            out.append(f"  --- plan v{v.version} ---")
            if v.plan is None:
                out.append(f"      INVALID: {v.invalid_reason}")
                continue
            for s in v.plan.steps:
                res = next((r for r in v.results if r.n == s.n), None)
                mark = ("      " if res is None
                        else ("  ok  " if res.ok else f" ERR  "))
                detail = "" if res is None or res.ok else f"({res.error})"
                out.append(f"   {mark}{s.n}. {s.tool}"
                           f"({json.dumps(s.args)}) {detail}")
            if v.critique:
                flag = " STRUCTURAL" if v.critique.structural else ""
                out.append(f"      critic: goal_met={v.critique.goal_met}"
                           f" revise={v.critique.revise}{flag}")
                for p in v.critique.problems:
                    out.append(f"        - {p}")
        out.append(f"  stop: {self.stop_reason}")
        out.append(f"  plans: {len(self.versions)}  "
                   f"re-plans: {self.replans}  tokens: {self.total_tokens}")
        return "\n".join(out)


# ---------------------------------------------------------------------------
# 4. Failure detectors
# ---------------------------------------------------------------------------

def detect_oscillation(trace: PlanTrace) -> str | None:
    """Has the planner produced a plan it already tried?

    Oscillation is the planning-era version of the Week 4 no-progress
    detector, and it is why that detector had to be built at the loop level
    rather than inside any single step.
    """
    seen: dict[str, int] = {}
    for v in trace.versions:
        sig = v.signature
        if sig in seen:
            return (f"plan v{v.version} repeats v{seen[sig]} exactly "
                    f"({len(v.plan.steps) if v.plan else 0} steps)")
        seen[sig] = v.version
    return None


def goal_drift(goal: "Goal", plan: Plan) -> list[str]:
    """Which parts of the goal does this plan no longer address?

    Goal drift is quiet: each re-plan looks locally reasonable while the
    plan as a whole stops serving the request. You cannot see it from one
    version; you have to compare against the ORIGINAL goal every time.
    """
    covered = {s.tool for s in plan.steps}
    return [need for need, tools in goal.requires.items()
            if not (set(tools) & covered)]


def score_plan(goal: "Goal", plan: Plan) -> dict[str, Any]:
    """Plan quality against the gold step set, measured before execution."""
    proposed = [s.tool for s in plan.steps]
    unknown = [t for t in proposed if t not in TOOLS]
    gold = goal.gold_tools
    hit = len(gold & set(proposed))
    return {"steps": len(proposed),
            "gold_covered": hit / max(len(gold), 1),
            "hallucinated_tools": unknown,
            "missing": sorted(gold - set(proposed)),
            "extra": sorted(set(proposed) - gold - set(unknown))}


# ---------------------------------------------------------------------------
# 5. The goals
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Goal:
    goal_id: str
    text: str
    gold_tools: set[str]
    requires: dict[str, list[str]]      # need -> tools that satisfy it
    note: str


GOALS: list[Goal] = [
    Goal("G1",
         "My order A1091 is very late. I want to know where it is, whether "
         "I'm owed anything, and I'd like it sent to my work address instead.",
         {"track_order", "get_policy", "request_approval",
          "check_address_changeable"},
         {"locate": ["track_order"],
          "remedy": ["get_policy", "request_approval"],
          "address": ["check_address_changeable"]},
         "Three needs in one message. Tests decomposition and coverage. "
         "A1091 is 14 days late, so the credit qualifies."),

    Goal("G2",
         "Order A1080 arrived a day late and I'd like compensation.",
         {"track_order", "get_policy"},
         {"locate": ["track_order"], "policy": ["get_policy"]},
         "One day late, below the 3-day threshold. The correct plan gathers "
         "evidence and then does NOT propose a credit. Tests whether the "
         "planner can plan its way to 'no'."),

    Goal("G3",
         "My order A1032 is late AND I think I've been charged twice for it.",
         {"track_order", "get_policy", "request_approval",
          "escalate_to_billing"},
         {"delivery": ["track_order", "get_policy"],
          "billing": ["escalate_to_billing"]},
         "Two issues, ONE of which is out of remit. Tests whether the plan "
         "splits them rather than trying to resolve both."),

    Goal("G4",
         "Order A9999 hasn't turned up and I want a refund today.",
         {"track_order", "escalate_to_human"},
         {"locate": ["track_order"], "handoff": ["escalate_to_human"]},
         "The order does not exist and there is no refund tool at all. Every "
         "plan will fail at step 1. STRUCTURAL: no amount of re-planning "
         "fixes a missing tool. Tests whether the critic says so."),
]


# ---------------------------------------------------------------------------
# 6. Prompt scaffolds
# ---------------------------------------------------------------------------

def _tool_catalogue() -> str:
    lines = []
    for name, spec in TOOLS.items():
        sig = ", ".join(spec.args) or ""
        lines.append(f"  {name}({sig})  [{spec.tier.value}]")
        lines.append(f"      {spec.description}")
    return "\n".join(lines)


PLANNER_SYSTEM = f"""You are the planner for Layla, a support agent at
Northwind Retail.

Given ONE customer message, produce a complete plan BEFORE anything runs.

TOOLS — you may use only these. There are no others.
{_tool_catalogue()}

RULES
- Gather evidence before proposing any remedy.
- request_approval is consequential: it may only appear after track_order
  and get_policy have both appeared earlier in the plan.
- Billing disputes are out of remit: plan to escalate them, never to
  resolve them.
- If the request needs a tool that does not exist, plan to escalate to a
  human instead of inventing one.
- Keep the plan as short as the goal allows.

Return ONE JSON object and nothing else:
{{"goal_restated": "...",
  "steps": [{{"n": 1, "tool": "track_order",
              "args": {{"order_id": "A1032"}},
              "why": "establish the delay"}}]}}"""


CRITIC_SYSTEM = """You are the critic. You did not write the plan and you
are not trying to be encouraging.

You receive the customer's goal, the plan that was tried, and what each step
actually returned. Decide three things.

1. goal_met — did the executed plan actually serve every part of the
   customer's request? Partial is not met.

2. structural — is the reason for failure something a NEW PLAN COULD NOT
   FIX? A missing tool, a permission that will always be denied, an order
   that does not exist, a policy threshold that is not met. If so, say so:
   re-planning cannot help and a human must take over.

3. revise — should we try a different plan? Only true when the failure is
   NOT structural and you can name what would change.

Return ONE JSON object and nothing else:
{"goal_met": false, "problems": ["..."], "structural": false,
 "revise": true}"""
'''

with open("lab7_kit.py", "w", encoding="utf-8") as f:
    f.write(kit_source)

import importlib, sys
sys.modules.pop("lab7_kit", None)
import lab7_kit as K
importlib.reload(K)

print(f"lab7_kit.py written: {len(kit_source.splitlines())} lines")
print("provider:", K.PROVIDER, "| model:", K.MODEL)
print("tools   :", list(K.TOOLS))
print("goals   :", [g.goal_id for g in K.GOALS])

lab7_kit.py written: 469 lines
provider: ollama | model: qwen2.5:3b
tools   : ['track_order', 'get_policy', 'check_address_changeable', 'request_approval', 'escalate_to_billing', 'escalate_to_human']
goals   : ['G1', 'G2', 'G3', 'G4']


### Prove the model answers, before anything depends on it

In [15]:
import json, re, time
from pydantic import ValidationError
from lab7_kit import (CRITIC_SYSTEM, PLANNER_SYSTEM, Critique, Plan,
                      PlanTrace, PlanVersion, Step, StepResult, Tier)

client = K.make_client()
r = client.chat.completions.create(
    model=K.MODEL, temperature=0, max_tokens=40,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}])
print("model says:", r.choices[0].message.content.strip())
print("tokens    :", r.usage.total_tokens)

model says: ready
tokens    : 38



## Part 1 — The plan contract (given)

A plan is a **typed artefact produced before anything runs**. That is the
whole difference from ReAct, and it is what makes the next part possible.

Read `Step` and `Plan`, then look at `signature()` — the identity of a
plan's *shape*, which is what makes oscillation detectable at all.

In [4]:
import inspect
print(inspect.getsource(K.Step))
print(inspect.getsource(K.Plan))

class Step(BaseModel):
    """One step of a plan. Note it is a PROPOSAL: nothing has run yet."""
    model_config = ConfigDict(extra="forbid")
    n: int = Field(ge=1, le=12)
    tool: str
    args: dict = Field(default_factory=dict)
    why: str = Field(max_length=200,
                     description="What this step establishes, in one line.")

class Plan(BaseModel):
    """A whole plan, produced before anything executes.

    THIS is what Plan-and-Execute buys you over ReAct: an artefact that
    exists before any action, and can therefore be inspected, validated,
    priced or shown to a human. ReAct has no such moment -- its Thought and
    Action fire in the same turn.
    """
    model_config = ConfigDict(extra="forbid")
    goal_restated: str = Field(max_length=300)
    steps: list[Step] = Field(min_length=1, max_length=12)

    def signature(self) -> str:
        """Identity of the plan's shape, for oscillation detection."""
        return "|".join(f"{s.tool}({json.dumps(s.arg

### The four goals

Each one has a different failure built in. Read the notes.

In [5]:
for g in K.GOALS:
    print(f"{g.goal_id}  {g.text}")
    print(f"     needs: {sorted(g.requires)}")
    print(f"     {g.note}\n")

G1  My order A1091 is very late. I want to know where it is, whether I'm owed anything, and I'd like it sent to my work address instead.
     needs: ['address', 'locate', 'remedy']
     Three needs in one message. Tests decomposition and coverage. A1091 is 14 days late, so the credit qualifies.

G2  Order A1080 arrived a day late and I'd like compensation.
     needs: ['locate', 'policy']
     One day late, below the 3-day threshold. The correct plan gathers evidence and then does NOT propose a credit. Tests whether the planner can plan its way to 'no'.

G3  My order A1032 is late AND I think I've been charged twice for it.
     needs: ['billing', 'delivery']
     Two issues, ONE of which is out of remit. Tests whether the plan splits them rather than trying to resolve both.

G4  Order A9999 hasn't turned up and I want a refund today.
     needs: ['handoff', 'locate']
     The order does not exist and there is no refund tool at all. Every plan will fail at step 1. STRUCTURAL: no amount

### The model seam, with validate-and-retry (given)

A 7B model will not reliably emit clean JSON. Same discipline as Week 3:
parse unrepaired first, count the repairs, hand validation errors back.

In [16]:
JSON_OBJ = re.compile(r"\{.*\}", re.S)
REPAIRS = {"unfenced": 0, "retries": 0, "gave_up": 0}


def _ask(system, user, max_tokens=700):
    r = client.chat.completions.create(
        model=K.MODEL, temperature=0, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}])
    return r.choices[0].message.content or "", r.usage.total_tokens


def _parse(raw, model_cls):
    obj = None
    try:
        obj = json.loads(raw)                 # unrepaired, and counted
    except json.JSONDecodeError:
        m = JSON_OBJ.search(raw)
        if m:
            REPAIRS["unfenced"] += 1
            try: obj = json.loads(m.group(0))
            except json.JSONDecodeError: obj = None
    if obj is None:
        return None, "not valid JSON"
    try:
        return model_cls.model_validate(obj), None
    except ValidationError as exc:
        e = exc.errors()[0]
        return None, f"{'.'.join(str(x) for x in e['loc'])}: {e['msg']}"


def propose(system, user, model_cls, tries=3):
    """Ask, validate, hand the error back. Bounded."""
    prompt, total = user, 0
    for _ in range(tries):
        raw, tok = _ask(system, prompt)
        total += tok
        obj, why = _parse(raw, model_cls)
        if obj is not None:
            return obj, None, total
        REPAIRS["retries"] += 1
        prompt = (f"{user}\n\nYour previous reply was rejected: {why}. "
                  "Return ONLY the corrected JSON object.")
    REPAIRS["gave_up"] += 1
    return None, why, total

print("client ready")

client ready


### See it plan, before any validation exists

Read the plan it produces. Is it in a sensible order? Did it invent a tool?

In [7]:
g1 = K.GOALS[0]
plan, why, tok = propose(PLANNER_SYSTEM,
                         f"CUSTOMER MESSAGE:\n{g1.text}", Plan)
if plan is None:
    print("planner failed:", why)
else:
    print(plan.goal_restated, "\n")
    for s in plan.steps:
        print(f"  {s.n}. {s.tool}({json.dumps(s.args)})  \u2014 {s.why}")
print(f"\ntokens: {tok}   repairs: {REPAIRS}")
print("\nNOTHING HAS RUN. That is the point of the next cell.")

Determine the status of order A1091, check if the customer is eligible for compensation, and verify if the delivery address can be changed. 

  1. track_order({"order_id": "A1091"})  — establish the delay and order status
  2. get_policy({})  — determine if the customer is eligible for compensation based on the late-delivery policy
  3. check_address_changeable({"order_id": "A1091"})  — verify if the delivery address can still be changed

tokens: 564   repairs: {'unfenced': 0, 'retries': 0, 'gave_up': 0}

NOTHING HAS RUN. That is the point of the next cell.



## Part 2 — Task 1: validate before executing

This is the checkpoint ReAct cannot give you. Every problem below is caught
with the model's work done and **nothing executed**.

> ### 🔧 Task 1
> Return a list of problems. Empty means the plan may run. Catch:
>
> - a step naming a tool that does not exist (**gate 1**)
> - a step missing a required argument (**gate 2**)
> - an `order_id` not matching `^A[0-9]{4}$` (**gate 2**)
> - `request_approval` appearing before *both* `track_order` and
>   `get_policy` (**gate 4**, as an ordering rule)

In [17]:
def validate_plan(plan: Plan) -> list[str]:
    problems = []
    completed_tools = set()

    for step in plan.steps:
        spec = K.TOOLS.get(step.tool)

        # Gate 1: the named tool must exist.
        if spec is None:
            problems.append(
                f"Step {step.n}: unknown tool '{step.tool}'"
            )
            continue

        # Gate 2: all required arguments must be present.
        for arg in spec.args:
            if arg not in step.args:
                problems.append(
                    f"Step {step.n}: '{step.tool}' is missing '{arg}'"
                )

        # Gate 2: validate any supplied order ID.
        if "order_id" in step.args:
            order_id = step.args["order_id"]
            if not isinstance(order_id, str) or not re.fullmatch(
                r"A[0-9]{4}", order_id
            ):
                problems.append(
                    f"Step {step.n}: invalid order_id {order_id!r}"
                )

        # Gate 4: evidence-gathering steps must precede approval.
        if step.tool == "request_approval":
            missing_evidence = {
                "track_order", "get_policy"
            } - completed_tools
            if missing_evidence:
                problems.append(
                    f"Step {step.n}: request_approval comes before "
                    f"{', '.join(sorted(missing_evidence))}"
                )

        completed_tools.add(step.tool)

    return problems


bad = Plan(goal_restated="test", steps=[
    Step(
        n=1,
        tool="issue_refund",
        args={"order_id": "A1032"},
        why="refund"
    ),
    Step(
        n=2,
        tool="request_approval",
        args={"order_id": "9999", "amount_percent": 10},
        why="credit"
    ),
])

print(validate_plan(bad))

["Step 1: unknown tool 'issue_refund'", "Step 2: invalid order_id '9999'", 'Step 2: request_approval comes before get_policy, track_order']


In [ ]:
# @title ✅ Solution — Task 1  { display-mode: "form" }


  step 1: no such tool 'issue_refund'
  step 2: '9999' is not a valid order id
  step 2: approval planned before evidence gathered



## Part 3 — Task 2: execute, with the gates still in front

The gates do not go away because you have a plan. A refused step is a
**result**, not an exception — the critic has to be able to read it.

> ### 🔧 Task 2
> Run the steps in order, returning one `StepResult` each. Refuse an unknown
> order, an approval with no evidence gathered, one below the threshold, and
> honour `allow_consequential`. Track what has succeeded: gate 4 depends on
> the run's history, which is why it cannot live in the plan.

In [18]:
def execute(plan: Plan, allow_consequential: bool = True) -> list[StepResult]:
    results = []
    tracked_orders = {}
    policy = None

    for step in plan.steps:
        spec = K.TOOLS.get(step.tool)
        tier = spec.tier.value if spec else None
        error = None
        observation = {}

        if spec is None:
            error = "unknown_tool"
        elif any(arg not in step.args for arg in spec.args):
            error = "missing_required_argument"
        elif "order_id" in step.args and step.args["order_id"] not in K.KNOWN_IDS:
            error = "order_not_found"
        elif spec.tier == Tier.CONSEQUENTIAL and not allow_consequential:
            error = "consequential_action_not_allowed"
        elif step.tool == "request_approval":
            order_id = step.args["order_id"]
            if order_id not in tracked_orders or policy is None:
                error = "evidence_missing"
            elif tracked_orders[order_id]["days_late"] < policy["threshold_days"]:
                error = "below_credit_threshold"

        if error is None:
            try:
                observation = spec.fn(**step.args)
                if not observation.get("ok", False):
                    error = observation.get("error", "tool_failed")
            except (TypeError, ValueError) as exc:
                error = f"invalid_arguments: {exc}"

        result = StepResult(
            n=step.n,
            tool=step.tool,
            args=step.args,
            tier=tier,
            ok=error is None,
            error=error,
            observation=observation,
        )
        results.append(result)

        # Only successful observations count as evidence for later steps.
        if result.ok and step.tool == "track_order":
            tracked_orders[step.args["order_id"]] = observation
        elif result.ok and step.tool == "get_policy":
            policy = observation

    return results


good = Plan(goal_restated="t", steps=[
    Step(n=1, tool="track_order",
         args={"order_id": "A1091"}, why="locate"),
    Step(n=2, tool="get_policy", args={}, why="threshold"),
    Step(n=3, tool="request_approval",
         args={"order_id": "A1091", "amount_percent": 10},
         why="remedy"),
])

for r in execute(good):
    print(
        f"  {r.n}. {r.tool:<22} "
        f"tier={r.tier:<14} ok={r.ok} {r.error or ''}"
    )

  1. track_order            tier=read           ok=True 
  2. get_policy             tier=read           ok=True 
  3. request_approval       tier=consequential  ok=True 


In [ ]:
# @title ✅ Solution — Task 2  { display-mode: "form" }


  1. track_order            tier=read           ok=True 
  2. get_policy             tier=read           ok=True 
  3. request_approval       tier=consequential  ok=True 



## Part 4 — Task 3: the loop

> ### 🔧 Task 3
> Implement `run`. Order matters:
>
> 1. **Validate before executing** — that is the whole point
> 2. Check `K.detect_oscillation` *before* spending another plan
> 3. Check `K.goal_drift` against the **original** goal every version
> 4. A **structural** critique means stop and escalate, not re-plan
> 5. Every exit sets `trace.stop_reason` — never a silent exit
>
> One question to settle while writing it: if the critic says `goal_met` but
> `goal_drift` disagrees, **which do you believe?**

In [19]:
def run(goal, max_versions: int = 3, allow_consequential: bool = True):
    """Plan, validate, execute, critique, and re-plan within a fixed limit."""
    trace = PlanTrace(goal_id=goal.goal_id)
    feedback = ""

    for version in range(1, max_versions + 1):
        planner_input = f"CUSTOMER MESSAGE:\n{goal.text}"
        if feedback:
            planner_input += (
                "\n\nPREVIOUS ATTEMPT AND FEEDBACK:\n"
                + feedback
                + "\nProduce a revised complete plan for the ORIGINAL message."
            )

        plan, why, plan_tokens = propose(
            PLANNER_SYSTEM, planner_input, Plan
        )
        current = PlanVersion(
            version=version,
            plan=plan,
            tokens=plan_tokens,
        )
        trace.versions.append(current)

        if plan is None:
            current.invalid_reason = why
            trace.stop_reason = "planner could not produce a valid plan"
            break

        # Check the proposal before any tool runs.
        problems = validate_plan(plan)
        if problems:
            current.invalid_reason = "; ".join(problems)
            trace.stop_reason = "plan failed validation before execution"
            break

        # Detect a repeated plan before executing it again.
        repeated = K.detect_oscillation(trace)
        if repeated:
            trace.stop_reason = f"oscillation: {repeated}"
            break

        # Always compare with the customer's ORIGINAL goal.
        drift = K.goal_drift(goal, plan)
        if drift:
            current.invalid_reason = "goal drift: " + ", ".join(drift)
            trace.stop_reason = current.invalid_reason
            break

        current.results = execute(
            plan,
            allow_consequential=allow_consequential,
        )

        critique_input = (
            f"ORIGINAL CUSTOMER GOAL:\n{goal.text}\n\n"
            f"PLAN:\n{plan.model_dump_json(indent=2)}\n\n"
            f"EXECUTION RESULTS:\n"
            f"{json.dumps([r.__dict__ for r in current.results], indent=2)}"
        )

        critique, critic_error, critic_tokens = propose(
            CRITIC_SYSTEM, critique_input, Critique
        )
        current.tokens += critic_tokens

        if critique is None:
            trace.stop_reason = (
                f"critic could not produce a valid verdict: {critic_error}"
            )
            break

        current.critique = critique

        if critique.structural:
            trace.stop_reason = "structural failure — escalate to a human"
            break

        if critique.goal_met:
            trace.stop_reason = "goal met"
            trace.answer = "Goal completed according to the critic."
            break

        if not critique.revise:
            trace.stop_reason = "critic declined further revision"
            break

        feedback = (
            "Plan: " + plan.model_dump_json()
            + "\nResults: "
            + json.dumps([r.__dict__ for r in current.results])
            + "\nCritic problems: "
            + "; ".join(critique.problems)
        )

    if not trace.stop_reason:
        trace.stop_reason = "maximum plan versions reached"

    return trace


trace = run(K.GOALS[0], max_versions=1)
print(trace.render())

goal G1
  --- plan v1 ---
     ok  1. track_order({"order_id": "A1091"}) 
     ok  2. get_policy({}) 
     ok  3. check_address_changeable({"order_id": "A1091"}) 
     ok  4. escalate_to_human({"reason": "customer wants to know about the order status, potential compensation, and address change options"}) 
      critic: goal_met=False revise=True STRUCTURAL
        - The order is 14 days late, which exceeds the policy threshold of 3 days and requires supervisor approval for a 10% credit. The tool 'get_policy' did not return a solution for the customer's request as it only provides information about the policy and not the actual compensation process. The order can be changed to the work address, but the customer needs to be informed about the policy and the need for supervisor approval for the credit. The plan cannot provide the customer with the information they need for the credit and supervisor approval.
        - The order is 14 days late, which exceeds the policy threshold of 3 days

I expect G1 to need re-planning because it has three needs. G4 should stop as a structural failure because the order is unknown and no refund tool exists. Extra attempts may increase token cost without improving the outcome.

In [ ]:
# @title ✅ Solution — Task 3  { display-mode: "form" }


goal G1
  --- plan v1 ---
     ok  1. track_order({"order_id": "A1091"}) 
     ok  2. get_policy({}) 
     ok  3. check_address_changeable({"order_id": "A1091"}) 
      critic: goal_met=False revise=True
        - The plan did not include steps to determine if the customer is owed anything based on the order's lateness.
        - The plan did not include sending the order to the work address.
  --- plan v2 ---
         1. track_order({"order_id": "A1091"}) 
         2. get_policy({}) 
         3. check_address_changeable({"order_id": "A1091"}) 
  stop: capped: oscillating — plan v2 repeats v1 exactly (3 steps)
  plans: 2  re-plans: 1  tokens: 1684



## Part 5 — Task 4: measure

**Predict the table before you run it.** Which goal needs most re-plans?
Which costs most? Which should stop without ever succeeding?

In [20]:
def measure(max_versions: int = 3):
    header = (
        f"{'Goal':<6} {'Gold coverage':<15} "
        f"{'Re-plans':<10} {'Tokens':<9} Stop reason"
    )
    print(header)
    print("-" * len(header))

    traces = []

    for goal in K.GOALS:
        trace = run(goal, max_versions=max_versions)
        traces.append(trace)

        first_plan = next(
            (v.plan for v in trace.versions if v.plan is not None),
            None,
        )

        if first_plan is None:
            coverage = "N/A"
        else:
            quality = K.score_plan(goal, first_plan)
            coverage = f"{quality['gold_covered']:.0%}"

        print(
            f"{goal.goal_id:<6} "
            f"{coverage:<15} "
            f"{trace.replans:<10} "
            f"{trace.total_tokens:<9} "
            f"{trace.stop_reason}"
        )

    return traces


measurement_traces = measure(max_versions=1)

Goal   Gold coverage   Re-plans   Tokens    Stop reason
-------------------------------------------------------
G1     100%            0          1913      structural failure — escalate to a human
G2     100%            0          1569      structural failure — escalate to a human
G3     50%             0          602       goal drift: billing
G4     50%             0          615       goal drift: handoff


In [ ]:
# @title ✅ Solution — Task 4  { display-mode: "form" }


goal    plans  re-plans   tokens   gold  stop
-------------------------------------------------------------------
G1          2         1     1686   75%  capped: oscillating — plan v2 repeats v1 exa
       (334s)
G2          3         2     3298  100%  structural: escalate, re-planning cannot hel
       (813s)
G3          3         2     3701   75%  capped: 3 plan versions
       (992s)
G4          1         0     1069   50%  structural: escalate, re-planning cannot hel
       (316s)

repairs: {'unfenced': 0, 'retries': 0, 'gave_up': 0}



## Part 6 — Exercises

The assessed part. Nothing here is scripted — the model is real, so report
what happened even when it is not what the note predicts.

### Exercise 1 — Planning your way to "no"

`A1080` is one day late; the threshold is three. A correct plan gathers
evidence and then **does not** propose a credit.

In [21]:
t = measurement_traces[1]
print(t.render())

# Q1. Did it propose the approval anyway? Which gate refused it?
# Q2. If it did NOT propose one, did it read the policy first, or guess
#     right? A right answer for the wrong reason is still a finding.

goal G2
  --- plan v1 ---
     ok  1. track_order({"order_id": "A1080"}) 
     ok  2. get_policy({}) 
     ok  3. check_address_changeable({"order_id": "A1080"}) 
     ok  4. escalate_to_human({"reason": "unable to resolve without further investigation"}) 
      critic: goal_met=False revise=False STRUCTURAL
        - The order is only 1 day late, which is below the threshold of 3+ working days for compensation. The policy only allows for a 10% credit if the order is 3+ days late, and this threshold cannot be changed as it is a structural policy setting.
  stop: structural failure — escalate to a human
  plans: 1  re-plans: 0  tokens: 1569


### Exercise 2 — Two issues, one out of remit

`G3` is a late delivery **and** a billing dispute. Support handles the
first; billing handles the second.

In [22]:
t = measurement_traces[2]
print(t.render())
v1 = next(v for v in t.versions if v.plan)
print("\nplan quality:", K.score_plan(K.GOALS[2], v1.plan))

# Q1. Did the plan split the two issues, or try to resolve both?
# Q2. escalate_to_billing exists. Did the planner find it?

goal G3
  --- plan v1 ---
         1. track_order({"order_id": "A1032"}) 
         2. get_policy({}) 
         3. check_address_changeable({"order_id": "A1032"}) 
         4. escalate_to_human({"reason": "order_status_and_double_charging"}) 
  stop: goal drift: billing
  plans: 1  re-plans: 0  tokens: 602

plan quality: {'steps': 4, 'gold_covered': 0.5, 'hallucinated_tools': [], 'missing': ['escalate_to_billing', 'request_approval'], 'extra': ['check_address_changeable', 'escalate_to_human']}


### Exercise 3 — The structural one

`G4` asks for a refund on an order that does not exist, and **there is no
refund tool at all**. Every plan fails at step 1, identically.

In [23]:
t = measurement_traces[3]
print(t.render())

# Q1. How many rounds did it spend before stopping? Each is a planner call
#     plus a critic call.
# Q2. Did the critic say STRUCTURAL, or did it keep suggesting revisions?
# Q3. Reflection on "the tool does not exist" produces a more eloquent way
#     of not having the tool. What should the agent have done instead?

goal G4
  --- plan v1 ---
         1. track_order({"order_id": "A9999"}) 
         2. get_policy({}) 
         3. check_address_changeable({"order_id": "A9999"}) 
         4. request_approval({"order_id": "A9999", "amount_percent": 100}) 
         5. escalate_to_billing({"order_id": "A9999", "description": "Customer request for refund due to delayed delivery"}) 
  stop: goal drift: handoff
  plans: 1  re-plans: 0  tokens: 615


### Exercise 4 — Goal drift

`G1` asks for three things: locate the order, check the remedy, change the
address. Watch whether a re-plan quietly drops one.

In [24]:
t = measurement_traces[0]
print(t.render())
print()
for v in t.versions:
    if v.plan:
        print(f"v{v.version} drift: {K.goal_drift(K.GOALS[0], v.plan)}")

# Q1. Did any version drop a requirement while fixing another?
# Q2. Did the CRITIC notice, or only goal_drift()?
# Q3. goal.requires was written by a human at design time. Why can the
#     agent not be trusted to write its own goal test?

goal G1
  --- plan v1 ---
     ok  1. track_order({"order_id": "A1091"}) 
     ok  2. get_policy({}) 
     ok  3. check_address_changeable({"order_id": "A1091"}) 
     ok  4. request_approval({"order_id": "A1091", "amount_percent": 10}) 
     ok  5. escalate_to_human({"reason": "order is late and cannot be delivered to the original address"}) 
      critic: goal_met=False revise=True STRUCTURAL
        - The order is 14 days late and the policy only covers orders 3+ days late. The tool for changing the delivery address did not return a changeable status for the order, indicating it is still at the depot and cannot be changed.
  stop: structural failure — escalate to a human
  plans: 1  re-plans: 0  tokens: 1913

v1 drift: []


### Exercise 5 — What reflection costs

Compare one attempt against three.

In [27]:
for cap in (1, 3):
    REPAIRS.update({"unfenced": 0, "retries": 0, "gave_up": 0})
    tot = 0
    for g in K.GOALS:
        tot += run(g, max_versions=cap).total_tokens
    print(f"max_versions={cap}: {tot} tokens across four goals")

# Q1. What multiple is three rounds over one?
# Q2. Did the extra rounds change any OUTCOME, or only the token count?
# Q3. G4 consumed its whole budget and returned nothing. Price that.

APIConnectionError: Connection error.

### Stretch — a cheaper critic

Your critic is the same model that wrote the plan, with a different prompt.
That is not an independent check.

Replace it with a deterministic goal test for one goal — a function that
inspects the results and decides — and compare cost and verdicts.

In [ ]:
def deterministic_critic(goal, plan, results) -> Critique:
    """Stretch: no model call at all.

    Hint: goal.requires plus the results is enough for most of what the
    model critic was guessing at.
    """
    raise NotImplementedError("stretch")

In [28]:
import ast
from pathlib import Path

names = [
    "_ask", "_parse", "propose",
    "validate_plan", "execute", "run", "measure"
]
found = {}

for cell_source in get_ipython().history_manager.input_hist_raw:
    try:
        tree = ast.parse(cell_source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name in names:
            found[node.name] = ast.get_source_segment(cell_source, node)

missing = set(names) - set(found)
if missing:
    raise RuntimeError(f"Missing functions: {sorted(missing)}")

header = '''"""COSC726 Week 7 Lab 6: planning, validation, execution, and measurement."""

import os
import re
import json
from pydantic import ValidationError

os.environ.setdefault("OLLAMA_MODEL", "qwen2.5:3b")

import lab7_kit as K
from lab7_kit import (
    CRITIC_SYSTEM, PLANNER_SYSTEM, Critique, Plan,
    PlanTrace, PlanVersion, Step, StepResult, Tier
)

client = K.make_client()
JSON_OBJ = re.compile(r"\\\\{.*\\\\}", re.S)
REPAIRS = {"unfenced": 0, "retries": 0, "gave_up": 0}

'''

source = header + "\n\n\n".join(found[name] for name in names) + "\n"
Path("planner.py").write_text(source, encoding="utf-8")

print("Created planner.py")
print("Functions:", ", ".join(names))

Created planner.py
Functions: _ask, _parse, propose, validate_plan, execute, run, measure


In [30]:
!python -m py_compile planner.py

In [31]:
from pathlib import Path

memo = """# COSC726 Week 7 Lab 6 — Decision Memo

Model used: qwen2.5:3b through Ollama.
Measurement limit: one plan version per goal.

## Measurement table

| Goal | First-plan gold coverage | Re-plans | Tokens | Stop reason |
|---|---:|---:|---:|---|
| G1 | 100% | 0 | 1913 | Critic labelled failure structural |
| G2 | 100% | 0 | 1569 | Critic labelled failure structural |
| G3 | 50% | 0 | 602 | Goal drift: billing |
| G4 | 50% | 0 | 615 | Goal drift: handoff |

Total measured tokens: 4699. Re-plans were zero because this
measurement allowed only one plan version per goal.

## 1. What did planning buy over ReAct?

The complete plan could be checked before tools ran. The validator
detected an invented refund tool, an invalid order ID, and approval
scheduled before evidence gathering. In G3 and G4, the goal-drift
detector stopped incomplete plans before any step executed. Without
this checkpoint, a step-by-step agent could attempt an invalid action
before discovering the problem.

## 2. What did it cost?

The one-version measurement used 4699 tokens across four goals.
Planning also committed the agent to steps before it saw tool results;
G2 included unnecessary address and escalation steps. A comparison
with three permitted versions did not complete within the available
runtime, so this run does not establish a measured cost multiplier.

## 3. Where did reflection help, and where did it not?

No successful re-plan was observed in this measurement because it
allowed one version per goal. G4 stopped for missing handoff coverage
before the order lookup ran, so this trace did not test whether the
critic would recognize the nonexistent order or missing refund tool.
A missing tool and nonexistent order cannot be repaired merely by
asking the same model to re-plan.

## 4. What did the detectors catch that the critic missed?

Goal drift detected that G3 omitted the billing handoff and G4 omitted
a human handoff before execution. G1 had no detected drift, but its
critic incorrectly called the outcome structural: the order was 14
days late, above the three-day threshold, and request_approval was
available and executed. The critic also misread the address result.
This shows that the critic's verdict needs independent checks.

## 5. Where does the agent still trust something it should not?

The loop trusts the model critic's structural verdict even when it
contradicts tool observations. The plan validator checks required
arguments and order ID format, but does not fully validate the
meaning of a requested amount: G4 proposed a 100% credit in response
to a refund request. Tool outputs and authorization must be checked
before treating a proposed action as justified.

## 6. What did this lab not tell you?

There were only four hand-written goals, one model, and one measured
run per goal, so there is no variance estimate or evidence of
generalization. The three-version comparison did not complete.
Gold tool coverage measures proposed tool names, not customer outcome.
The model critic is not an independent authority and made factual
errors in this run. Results may change with another model, prompt,
runtime, or sample of goals.
"""

Path("decision_memo.md").write_text(memo, encoding="utf-8")
print("Created decision_memo.md")

Created decision_memo.md



## Submit

- this notebook, executed
- `planner.py` — your validator, executor and loop
- your measurement table
- `decision_memo.md`

### The decision memo

1. **What did planning buy over ReAct?** Name the failures `validate_plan`
   caught with nothing executed, and what each would have cost in a ReAct
   loop.
2. **What did it cost?** Tokens and adaptivity. Quote your numbers.
3. **Where did reflection help, and where did it not?** G4 is the test.
4. **What did the detectors catch that the critic missed?** Look hard at
   drift on G1.
5. **Where does your agent still trust something it should not?**
6. **What did this lab not tell you?**

Questions 3 and 4 carry the most marks. For question 6 be specific: four
goals written by one person, one model, one run each — no variance estimate.
And note who your critic is.

